In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

from sklearn.linear_model import Perceptron #used for simple linear classification task

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from tensorflow.keras.models import Sequential #To create neural network layer by layer

from tensorflow.keras.layers import Conv2D #Conv2D extract features(Conolutional Layer)
from tensorflow.keras.layers import Dense # Creates fully connected (Dense) layers, used for hidden or output layers
from tensorflow.keras.layers import Flatten #Flatten 2D matrics into 1D array
from tensorflow.keras.layers import MaxPooling2D # Reduce image size
from tensorflow.keras.layers import Dropout # To prevent overfitting

from tensorflow.keras.utils import to_categorical # Convert numeric class into OneHotEncoding
from tensorflow import keras

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(60000, 28, 28)
(60000,)
(10000, 28, 28)
(10000,)


In [ ]:
# Converting values between 0 to 1 -> It is better approach to convert into 0 - 1
X_train = X_train.astype('float')/255.0
X_test = X_test.astype('float')/255.0

In [ ]:
#Reshaping into array
#If your images were flattened into 784 pixel values per image, their shape would be (60000, 784). You could reshape them back into images like this:
# Reshape training images into (number_of_images, 28 rows, 28 columns).
# -1 automatically calculates the number of images.

X_train_img = X_train.reshape(-1, 28, 28) # -1 is 1st column and converted into 28*28 metrics
X_test_img = X_test.reshape(-1, 28, 28)

In [ ]:
y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

In [ ]:
print(y_train_cat.shape)

(60000, 10)


In [ ]:
perceptron = Sequential([
    Flatten(input_shape=(28,28)),
    Dense(10,activation='softmax') # Output layer and we have 10 categories
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [ ]:
perceptron.compile(optimizer='sgd',loss='categorical_crossentropy',metrics=['accuracy'])

In [ ]:
from re import VERBOSE
from sklearn.utils import validation
history = perceptron.fit(X_train_img,y_train_cat,epochs=5,batch_size=5,
                         validation_data=(X_test_img,y_test_cat),
                         verbose=1)

Epoch 1/5
12000/12000 ━━━━━━━━━━━━━━━━━━━━ 27s 2ms/step - accuracy: 0.8783 - loss: 0.4525 - val_accuracy: 0.9067 - val_loss: 0.3286
Epoch 2/5
12000/12000 ━━━━━━━━━━━━━━━━━━━━ 42s 2ms/step - accuracy: 0.9073 - loss: 0.3278 - val_accuracy: 0.9168 - val_loss: 0.3027
Epoch 3/5
12000/12000 ━━━━━━━━━━━━━━━━━━━━ 41s 2ms/step - accuracy: 0.9139 - loss: 0.3074 - val_accuracy: 0.9185 - val_loss: 0.2924
Epoch 4/5
12000/12000 ━━━━━━━━━━━━━━━━━━━━ 26s 2ms/step - accuracy: 0.9170 - loss: 0.2963 - val_accuracy: 0.9201 - val_loss: 0.2855
Epoch 5/5
12000/12000 ━━━━━━━━━━━━━━━━━━━━ 28s 2ms/step - accuracy: 0.9187 - loss: 0.2892 - val_accuracy: 0.9212 - val_loss: 0.2822


In [ ]:
acc_percp = perceptron.evaluate(X_test_img, y_test_cat, verbose=0)[1]
acc_percp

0.9211999773979187

In [ ]:
#ANN
ann = Sequential([
    Flatten(input_shape=(28,28)),
    Dense(128, activation="relu"),
    Dense(64, activation="relu"),
    Dense(10, activation="softmax")
])

In [ ]:
ann.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [ ]:
history_ann = ann.fit(X_train_img, y_train_cat, epochs=5, batch_size=32, validation_data=(X_test_img, y_test_cat), verbose=1)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.9299 - loss: 0.2371 - val_accuracy: 0.9632 - val_loss: 0.1201
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 5ms/step - accuracy: 0.9696 - loss: 0.0990 - val_accuracy: 0.9673 - val_loss: 0.1005
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.9777 - loss: 0.0724 - val_accuracy: 0.9713 - val_loss: 0.0876
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 10s 5ms/step - accuracy: 0.9830 - loss: 0.0536 - val_accuracy: 0.9773 - val_loss: 0.0770
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 8s 4ms/step - accuracy: 0.9862 - loss: 0.0418 - val_accuracy: 0.9767 - val_loss: 0.0791


In [ ]:
acc_ann = ann.evaluate(X_test_img, y_test_cat, verbose=0)[1]
acc_ann

0.9767000079154968

In [ ]:
X_train_cnn = X_train.reshape(-1, 28, 28,1)
X_test_cnn = X_test.reshape(-1, 28, 28, 1)

In [ ]:
cnn = Sequential([
    Conv2D(32, kernel_size=(3,3), activation="relu", input_shape=(28,28,1)), #Here 1 is channel (1 for BW 3 for RGB)
    MaxPooling2D(pool_size=(2,2)),
    Conv2D(64, kernel_size=(3,3), activation="relu"),
    MaxPooling2D(pool_size=(2,2)),
    Flatten(),
    Dense(128, activation="relu"),
    Dropout(0.5),
    Dense(10, activation="softmax")
])

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
cnn.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

In [ ]:
history_cnn = cnn.fit(X_train_cnn, y_train_cat, epochs=5, batch_size=32, validation_data=(X_test_cnn, y_test_cat), verbose=1)

Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 66s 34ms/step - accuracy: 0.9305 - loss: 0.2254 - val_accuracy: 0.9816 - val_loss: 0.0525
Epoch 2/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 61s 32ms/step - accuracy: 0.9746 - loss: 0.0861 - val_accuracy: 0.9874 - val_loss: 0.0375
Epoch 3/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 80s 31ms/step - accuracy: 0.9814 - loss: 0.0647 - val_accuracy: 0.9903 - val_loss: 0.0286
Epoch 4/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 60s 32ms/step - accuracy: 0.9845 - loss: 0.0506 - val_accuracy: 0.9893 - val_loss: 0.0338
Epoch 5/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 61s 33ms/step - accuracy: 0.9872 - loss: 0.0425 - val_accuracy: 0.9902 - val_loss: 0.0303


In [ ]:
acc_cnn = cnn.evaluate(X_test_cnn, y_test_cat, verbose=0)[1]
acc_cnn

0.9901999831199646